## Data Source

This notebook requires the historical pricing dataset, which is not included in this repository due to GitHub's 100 MB file size limit.

To reproduce this step:
1. Download the dataset [here](https://traderjoesprices.com/)
2. Place it in `data/raw/historical_prices.csv`
3. Run this notebook

In [ ]:
# Imports
import pandas as pd
from pathlib import Path

In [ ]:
# Load CSVs

DATA = Path("../data/raw")
MAPS = Path("../mappings")


reviews = pd.read_csv(DATA / "reviews.csv")
prices = pd.read_csv(DATA / "historical_prices.csv", low_memory=False)

category_map = pd.read_csv(MAPS / "category_mapping.csv")
price_fixes = pd.read_csv(MAPS / "manual_price_fixes.csv")
category_overrides = pd.read_csv(MAPS / "product_category_overrides.csv")


In [ ]:
# Initial review of reviews dataset

missing_prices = reviews["Price (USD)*"].isna().sum()
missing_scores = reviews["Score (?/10)"].isna().sum()


overview = f"""
Initial dataset
---------------
Rows: {len(reviews)}
Columns: {len(reviews.columns)}

Missing values:
- Prices: {missing_prices}
- Scores: {missing_scores}

Columns:
{"\n".join(f"- {col}" for col in reviews.columns)}
"""

print(overview)


In [ ]:
# Initial review of prices
overview = f"""
Trade Joe's Prices dataset
---------------
Rows: {len(prices)}
Columns: {len(prices.columns)}

Number of missing values: {prices.isna().any().sum()}

Columns:
{"\n".join(f"- {col}" for col in prices.columns)}
"""

print(overview)

In [ ]:
# Dropping unnecessary columns/rows
reviews = reviews.drop(
    columns=['Unnamed: 5', 'Unnamed: 6']
)
# This eliminates the missing value detected in our initial review
prices = prices.drop(
    columns=["sku", "store_code", "availability"]
)
prices = prices.drop(prices.index[0])

# Simplifying column names
reviews.columns = reviews.columns.str.lower().str.strip()
prices.columns = prices.columns.str.lower().str.strip()

reviews = reviews.rename(
    columns={
    'product name':'name', 
    'score (?/10)':'score', 
    'price (usd)*':'price'
    }
)

prices = prices.rename(
    columns={
        'item_title':'name',
        'retail_price':'price',
    }
)

# Verifying new columns
assert reviews.columns.tolist() == ['name', 'score', 'price', 'category', 'date']
assert prices.columns.tolist() == ['price', 'name', 'inserted_at']

In [ ]:
# Normalize text formatting across all datasets to prevent mismatches 
# during joins and category mapping

def normalize_text_cols(df):
    cols = df.select_dtypes(include="string").columns

    df[cols] = (
        df[cols]
        .apply(lambda s: s.str.strip().str.lower())
    )

    return df

# Applying string cleaning to all dataframes

reviews = normalize_text_cols(reviews)
prices = normalize_text_cols(prices)

category_map = normalize_text_cols(category_map)
price_fixes = normalize_text_cols(price_fixes)
category_overrides = normalize_text_cols(category_overrides)

In [ ]:
# Converting price data from dollar values to floats

reviews["price"] = (
    reviews["price"]
    .replace(r"\$", "", regex=True)
    .astype(float)
)

prices["price"] = pd.to_numeric(
    prices["price"],
    errors="coerce"
)

# Converting timestamp data

reviews["date"] = pd.to_datetime(
    reviews["date"]
)

prices["inserted_at"] = pd.to_datetime(
    prices["inserted_at"]
)

# Sorting by date

reviews = reviews.sort_values("date")
prices = prices.sort_values("inserted_at")

In [ ]:
# Checking no nulls exist in prices dataset

assert prices.isna().sum().any() == 0

In [ ]:
# Checking for null values in reviews

missing = reviews.isna().sum()
missing[missing > 0]

In the following merge, prices displayed will be the most recent product price available before the review date. Therefore, price data is matched according to timestamps.

In [ ]:
# Recovered missing product prices by joining historical product 
# catalog data

filled = pd.merge_asof(
    reviews,
    prices,
    left_on="date",
    right_on="inserted_at",
    by="name",
    direction="backward"
)

In [ ]:
# Find products with missing values, even after automation

count = len(filled[filled['price_y'].isnull() & filled['price_x'].isnull()])
print(f"{count} products are missing prices")

In [ ]:
# Fill in missing prices, part 2 (manual lookup)

filled = filled.merge(
    price_fixes,
    on="name",
    how="left",
)

In [ ]:
# Merge price columns

filled['price_x'] = filled['price_x'].fillna(
    filled['price_y']
)
filled['price_x'] = filled['price_x'].fillna(
    filled['price']
)

# Drop unecessary columns

filled = filled.drop(
    columns=['price_y','inserted_at','price']
)

# Rename back to price

filled = filled.rename(columns={'price_x' : 'price'})


In [ ]:
# Verifying for null values

missing = filled.isna().sum()
missing[missing > 0]

Note: Two reviews did not include scores because the reviewer did not provide a rating, however these rows were kept for future analysis.

In [ ]:
# Summarizing recovered missing prices

manual_recovered_prices = len(price_fixes)
auto_recovered_prices = missing_prices - manual_recovered_prices

result = f"""
Missing prices recovery summary
-----------------------
Initial missing prices: {missing_prices}
Recovered automatically: {auto_recovered_prices}
Recovered manually: {manual_recovered_prices}
Final missing prices: {filled["price"].isna().sum()}
"""

print(result)

In [ ]:
# Fixing error - Manual reassignment (typo in raw data)
# Items were marked as "microwavable (latin america)" in raw data

filled["category"] = (
    filled["category"]
    .replace({
        "microwavable (latin america)":
        "microwavable (latin american)"
    })
)

In [ ]:
# Applying general category mapping (re-categorizing products with two 
# new columns)

filled = filled.merge(
    category_map,
    on=["category"],
    how="left"
)

# Checking how many products still need to be categorized*

missing_st = len(filled[filled["storage_type"].isna()])
missing_fc = len(filled[filled["food_category"].isna()])
print(f"{missing_st} items have missing storage types")
print(f"{missing_fc} items have missing food categories")

*Some items' categories were not listed in the general category mapping, as the items in these categories were of mixed types. Therefore, a separate category override mapping was created for these items.

In [ ]:
# Applying product category overrides-
# Certain categories such as "Holiday Desserts" contain products from 
# multiple storage stypes, so a product-level lookup overrides general 
# category mapping

filled = filled.merge(
    category_overrides,
    on=["name"],
    how="left",
    suffixes=("","_new")
)

filled["storage_type"] = filled["storage_type"].fillna(
    filled["storage_type_new"]
)

filled["food_category"] = filled["food_category"].fillna(
    filled["food_category_new"]
)

filled = filled.drop(columns=["storage_type_new", "food_category_new"])

# Verifying no missing storage type and food category values

assert filled["storage_type"].isna().sum() == 0
assert filled["food_category"].isna().sum() == 0

In [ ]:
# Summarizing category re-organization coverage

num_rows = len(reviews)
final_missing = len(filled[filled["storage_type"].isna()])


result = f"""
Category mapping coverage summary
---------------------------------
General category mapping:
{num_rows - missing_st}/{num_rows} products ({((num_rows - missing_st) / num_rows):.1%})

After category overrides:
{num_rows - final_missing}/{num_rows} prdoucts ({((num_rows - final_missing) / num_rows):.1%})
"""

print(result)

In [ ]:
# Final review
overview = f"""
Final dataset
---------------
Rows: {len(filled)}
Columns: {len(filled.columns)}

Recovered values:
- Price lookup: {auto_recovered_prices}
- Manual price fixes: {manual_recovered_prices}

Columns:
{"\n".join(f"- {col}" for col in filled.columns)}
"""

print(overview)

In [ ]:
# Save to new CSV
filled.to_csv(
    "../data/processed/cleaned_tj_reviews.csv", 
    index=False
)